# Model and evaluate low autocorrelation binary sequences

This guide constructs a LABS spin model, checks a small solution exhaustively, and translates it to an Ising operator. All examples run locally and require no backend, credentials, or commercial solver.

For a sequence of $N$ spins $s_i\in\{-1,+1\}$, the aperiodic correlations are $C_k=\sum_{i=0}^{N-k-1}s_i s_{i+k}$ and the energy is $E=\sum_{k=1}^{N-1}C_k^2$. The merit factor is $N^2/(2E)$. Lower energy and higher merit factor are better.

References: [Packebusch and Mertens (2016)](https://arxiv.org/abs/1512.02475), [Shaydulin et al. (2024)](https://arxiv.org/abs/2308.02342).

In [ ]:
from itertools import product

import numpy as np
from qiskit.quantum_info import Statevector
from qiskit_addon_opt_mapper.applications import LABS
from qiskit_addon_opt_mapper.applications.labs import energy_distribution, labs_statistics
from qiskit_addon_opt_mapper.converters import SpinToBinary
from qiskit_addon_opt_mapper.translators import to_ising

## Construct a model

`LABS(num_bits=N)` creates an unconstrained minimization problem with $N$ spin variables. The quadratic and quartic objective follows from expanding the squared correlations and using $s_i^2=1$. Creating a model does not solve it.

In [ ]:
labs = LABS(num_bits=8)
problem = labs.to_optimization_problem()
print(problem.prettyprint())

## Verify a small solution

Exhaustive enumeration is practical here because there are only $2^8=256$ sequences. It is exponential in $N$ and should not be used as a large-instance solver. The reported minimum for length eight is 8 (Packebusch and Mertens, Table 1).

`interpret()` accepts spin results in model-variable order. It does not automatically interpret binary arrays. All spins must be exactly -1 or +1.

In [ ]:
best_spins = min(product((-1, 1), repeat=labs.num_bits), key=labs.energy)
sequence = labs.interpret(np.asarray(best_spins))
energy = labs.energy(sequence)
assert energy == 8
assert problem.objective.evaluate(sequence) == energy
print("Sequence:", sequence)
print("Energy:", energy)
print("Merit factor:", labs.merit_factor(sequence))

## Ising mapping and the constant offset

`to_ising(problem)` returns `(operator, offset)`. The objective equals the computational-basis expectation of `operator` **plus** `offset`. In contrast, `labs.cost_hamiltonian()` includes that constant as an identity term.

Spin $s_i$ corresponds to qubit $i$. Qiskit uses little-endian indexing: qubit zero is the least-significant integer bit and the rightmost character in a displayed bit string. Convert bits explicitly with $s_i=1-2b_i$.

In [ ]:
operator, offset = to_ising(problem)
bits = (1 - np.asarray(sequence)) // 2
state_index = sum(int(bit) << index for index, bit in enumerate(bits))
basis_state = Statevector.from_int(state_index, 2**labs.num_bits)
operator_energy = float(basis_state.expectation_value(operator).real) + offset
full_energy = float(basis_state.expectation_value(labs.cost_hamiltonian()).real)
assert np.isclose(operator_energy, energy)
assert np.isclose(full_energy, energy)
assert labs.interpret(1 - 2 * bits) == sequence
print("Separate offset:", offset)
print("Operator expectation + offset:", operator_energy)

## Binary model conversion

For a workflow requiring binary decision variables, `SpinToBinary` builds the converted model. Its `interpret()` reconstructs the original spins. Feed that reconstructed result to `labs.interpret()`.

In [ ]:
converter = SpinToBinary()
binary_problem = converter.convert(problem)
assert binary_problem.objective.evaluate(bits) == energy
restored_sequence = labs.interpret(converter.interpret(bits))
assert restored_sequence == sequence
print("Restored sequence:", restored_sequence)

## Exact and sampled energy statistics

`labs_statistics(N)` enumerates every sequence and returns a sorted energy histogram. Its runtime is $O(N^2 2^N)$; use it only for small instances.

The synthetic counts below stand in for a sampler result. `energy_distribution()` sums sample multiplicities by energy. Integer keys require an explicit width so leading zero bits are retained. The counts must contain only the logical LABS register; correct any routing permutation before using the helper.

In [ ]:
exact_histogram = labs_statistics(labs.num_bits)
assert sum(exact_histogram.values()) == 2**labs.num_bits
best_key = "".join(str(int(bit)) for bit in reversed(bits))
string_counts = {best_key: 20, "0" * labs.num_bits: 3}
integer_counts = {state_index: 20, 0: 3}
sampled_histogram = energy_distribution(string_counts, num_spins=labs.num_bits)
assert sampled_histogram == energy_distribution(integer_counts, num_spins=labs.num_bits)
assert sum(sampled_histogram.values()) == 23
print("Exact histogram:", exact_histogram)
print("Sampled histogram:", sampled_histogram)

## Input boundaries

Sequence lengths must be positive integers, and sequences must contain exactly the expected number of spins. For the single-spin instance the energy is zero and the merit factor is positive infinity. The spelling `evaluate_LABS` remains an alias for `evaluate_labs` for compatibility with existing research code.

In [ ]:
single = LABS(num_bits=1)
assert single.energy([1]) == 0
assert np.isinf(single.merit_factor([1]))